In [1]:
import os
import glob
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Filter out specific FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning, 
                       message="use_inf_as_na option is deprecated")
warnings.filterwarnings("ignore", category=FutureWarning, 
                       message="When grouping with a length-1 list-like")

from causal_evaluation import load_experiment_results, calculate_metrics, modified_load_experiment_results
from causal_visualization import (
    plot_metrics_comparison,
    plot_error_comparison,
    plot_effect_comparison,
    plot_effect_histogram
)

# Set the main study folder path
main_study_folder = "full_study_results_2025-04-02_13-48-37"

# Create a results folder for the analysis
results_folder = f"analysis_results_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
os.makedirs(results_folder, exist_ok=True)

print(f"Loading results from {main_study_folder}...")

# Load parameter studies
print("Loading autocorrelation study...")
auto_results = modified_load_experiment_results(os.path.join(main_study_folder, "auto_study"), "auto")
print("Loading cross-link study...")
cross_results = modified_load_experiment_results(os.path.join(main_study_folder, "cross_study"), "cross")
print("Loading noise study...")
noise_results = modified_load_experiment_results(os.path.join(main_study_folder, "noise_study"), "noise")

# Convert to DataFrames for easier analysis
def results_to_dataframe(results_dict, param_name):
    rows = []
    
    for param_value, effect_dict in results_dict.items():
        for effect_key, effect_data in effect_dict.items():
            # Extract data
            row = {
                'param_name': param_name,
                'param_value': param_value,
                'effect_key': effect_key,
                'true_effect': effect_data.get('true_effect'),
                'true_graph_effect': effect_data.get('true_graph_effect'),
                'pcmci_effect': effect_data.get('pcmci_effect'),
                'bagged_effect': effect_data.get('bagged_effect')
            }
            
            # Add bootstrap statistics if available
            bootstrap_stats = effect_data.get('bootstrap_stats', {})
            if bootstrap_stats:
                row.update({
                    'bootstrap_mean': bootstrap_stats.get('mean'),
                    'bootstrap_std': bootstrap_stats.get('std'),
                    'bootstrap_ci_lower': bootstrap_stats.get('ci_lower'),
                    'bootstrap_ci_upper': bootstrap_stats.get('ci_upper'),
                    'bootstrap_success_rate': bootstrap_stats.get('estimation_success_rate')
                })
            
            # Add bootstrap effects if available
            if 'bootstrap_effects' in effect_data:
                row['bootstrap_effects'] = effect_data['bootstrap_effects']
            
            # Add timing information if available
            timings = effect_data.get('timings', {})
            if timings:
                for method, time_val in timings.items():
                    row[f'{method}_time'] = time_val
            
            rows.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(rows)
    
    # Add error columns
    for method in ['pcmci', 'bagged', 'bootstrap']:
        if method == 'bootstrap':
            effect_col = 'bootstrap_mean'
        else:
            effect_col = f'{method}_effect'
        
        # Skip if column doesn't exist
        if effect_col not in df.columns:
            continue
        
        # Calculate errors - handle NaN values safely
        df[f'{method}_error'] = df[effect_col].subtract(df['true_effect'], fill_value=np.nan)
        df[f'{method}_abs_error'] = df[f'{method}_error'].abs()
        df[f'{method}_squared_error'] = df[f'{method}_error'].pow(2)
        
        # Flag if true effect is within CI (for bootstrap)
        if method == 'bootstrap' and 'bootstrap_ci_lower' in df.columns and 'bootstrap_ci_upper' in df.columns:
            df['ci_covers_true'] = (df['true_effect'] >= df['bootstrap_ci_lower']) & \
                                  (df['true_effect'] <= df['bootstrap_ci_upper'])
    
    return df
        


Loading results from full_study_results_2025-04-02_13-48-37...
Loading autocorrelation study...
Loading cross-link study...
Loading noise study...


In [2]:

# Convert to DataFrames
auto_df = results_to_dataframe(auto_results, "auto")
cross_df = results_to_dataframe(cross_results, "cross")
noise_df = results_to_dataframe(noise_results, "noise")

# Combine all results
all_results = pd.concat([auto_df, cross_df, noise_df], ignore_index=True)

# Save the processed data
all_results.to_csv(os.path.join(results_folder, "all_results.csv"), index=False)

# Print basic information
print(f"Loaded {len(all_results)} effect estimations")
print(f"Parameters: {all_results['param_name'].unique()}")
print(f"Parameter values: {sorted(all_results['param_value'].unique())}")
print(f"Effect pairs: {all_results['effect_key'].unique()}")

# Calculate metrics for each parameter study
auto_metrics = calculate_metrics(auto_df, group_by='param_value')
cross_metrics = calculate_metrics(cross_df, group_by='param_value')
noise_metrics = calculate_metrics(noise_df, group_by='param_value')

# Save metrics to CSV
auto_metrics.to_csv(os.path.join(results_folder, "auto_metrics.csv"), index=False)
cross_metrics.to_csv(os.path.join(results_folder, "cross_metrics.csv"), index=False)
noise_metrics.to_csv(os.path.join(results_folder, "noise_metrics.csv"), index=False)

Loaded 6 effect estimations
Parameters: ['auto' 'cross' 'noise']
Parameter values: [np.float64(0.01), np.float64(0.1), np.float64(0.99), np.float64(2.0)]
Effect pairs: ['0_-2_to_3_0']


In [3]:
calculate_metrics(auto_df, group_by='param_value')

,param_value,pcmci_mae,pcmci_rmse,pcmci_bias,bagged_mae,bagged_rmse,bagged_bias,bootstrap_mae,bootstrap_rmse,bootstrap_bias,bootstrap_ci_coverage
0,0.01,0.002982,0.002982,0.002982,0.007849,0.007849,0.007849,0.022710,0.022710,0.022710,1.0
1,0.99,0.722882,0.722882,-0.722882,0.662389,0.662389,-0.662389,0.501936,0.501936,-0.501936,0.0


In [4]:


# ------------------------------------------------------------------
# Generate plots
# ------------------------------------------------------------------
print("Generating MAE comparison plots...")

# Autocorrelation
fig = plot_metrics_comparison(
    auto_metrics, 
    param_name='param_value',
    metric='mae', 
    methods=['true_graph','pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "auto_mae_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_metrics_comparison(
    cross_metrics, 
    param_name='param_value',
    metric='mae', 
    methods=['true_graph','pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "cross_mae_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_metrics_comparison(
    noise_metrics, 
    param_name='param_value',
    metric='mae', 
    methods=['true_graph','pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "noise_mae_comparison.png")
)
plt.close(fig)

print("Generating RMSE comparison plots...")

# Autocorrelation
fig = plot_metrics_comparison(
    auto_metrics, 
    param_name='param_value',
    metric='rmse', 
    methods=['true_graph','pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "auto_rmse_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_metrics_comparison(
    cross_metrics, 
    param_name='param_value',
    metric='rmse', 
    methods=['true_graph','pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "cross_rmse_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_metrics_comparison(
    noise_metrics, 
    param_name='param_value',
    metric='rmse', 
    methods=['true_graph','pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "noise_rmse_comparison.png")
)
plt.close(fig)

print("Generating effect comparison plots...")

# Autocorrelation
fig = plot_effect_comparison(
    auto_df, 
    param_name='auto',
    save_path=os.path.join(results_folder, "auto_effect_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_effect_comparison(
    cross_df, 
    param_name='cross',
    save_path=os.path.join(results_folder, "cross_effect_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_effect_comparison(
    noise_df, 
    param_name='noise',
    save_path=os.path.join(results_folder, "noise_effect_comparison.png")
)
plt.close(fig)

print("Generating error comparison plots...")

# Autocorrelation
fig = plot_error_comparison(
    auto_df, 
    param_name='auto',
    error_type='abs',
    save_path=os.path.join(results_folder, "auto_error_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_error_comparison(
    cross_df, 
    param_name='cross',
    error_type='abs',
    save_path=os.path.join(results_folder, "cross_error_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_error_comparison(
    noise_df, 
    param_name='noise',
    error_type='abs',
    save_path=os.path.join(results_folder, "noise_error_comparison.png")
)
plt.close(fig)

print("Generating bias comparison plots...")

# Autocorrelation
fig = plot_error_comparison(
    auto_df, 
    param_name='auto',
    error_type='raw',
    save_path=os.path.join(results_folder, "auto_bias_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_error_comparison(
    cross_df, 
    param_name='cross',
    error_type='raw',
    save_path=os.path.join(results_folder, "cross_bias_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_error_comparison(
    noise_df, 
    param_name='noise',
    error_type='raw',
    save_path=os.path.join(results_folder, "noise_bias_comparison.png")
)
plt.close(fig)



Generating MAE comparison plots...
Generating RMSE comparison plots...
Generating effect comparison plots...
Generating error comparison plots...
Generating bias comparison plots...


In [5]:

# ------------------------------------------------------------------
# Plot CI coverage
# ------------------------------------------------------------------
print("Analyzing bootstrap confidence interval coverage...")

# Calculate CI coverage rates by parameter value
if 'ci_covers_true' in auto_df.columns:
    auto_ci_coverage = auto_df.groupby('param_value')['ci_covers_true'].mean()
    cross_ci_coverage = cross_df.groupby('param_value')['ci_covers_true'].mean()
    noise_ci_coverage = noise_df.groupby('param_value')['ci_covers_true'].mean()

    # Plot CI coverage
    plt.figure(figsize=(12, 8))
    plt.plot(auto_ci_coverage.index, auto_ci_coverage.values, marker='o', label='Autocorrelation')
    plt.plot(cross_ci_coverage.index, cross_ci_coverage.values, marker='s', label='Cross-link Strength')
    plt.plot(noise_ci_coverage.index, noise_ci_coverage.values, marker='^', label='Noise Level')
    plt.axhline(y=0.95, color='r', linestyle='--', label='Expected 95% coverage')
    plt.xlabel('Parameter Value')
    plt.ylabel('CI Coverage Rate')
    plt.title('Bootstrap 95% CI Coverage Rate')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(results_folder, "ci_coverage_analysis.png"), dpi=300, bbox_inches='tight')
    plt.close()

    # Save CI coverage data
    ci_coverage_df = pd.DataFrame({
        'param_value': auto_ci_coverage.index,
        'auto_coverage': auto_ci_coverage.values,
        'cross_coverage': cross_ci_coverage.values#,
     #   'noise_coverage': noise_ci_coverage.values
    })
    ci_coverage_df.to_csv(os.path.join(results_folder, "ci_coverage.csv"), index=False)

# ------------------------------------------------------------------
# Custom bootstrap histogram implementation since we have issues with the existing one
# ------------------------------------------------------------------
print("Generating bootstrap distribution examples...")

def plot_custom_bootstrap_histogram(bootstrap_effects, true_effect=None, title=None, save_path=None):
    """Plot histogram of bootstrap effect estimates handling None and non-numeric values"""
    plt.figure(figsize=(10, 6))
    
    # Filter out None values and convert to numeric
    if bootstrap_effects is None:
        plt.text(0.5, 0.5, "No bootstrap effects data available", 
                ha='center', va='center', fontsize=14)
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        return plt.gcf()
    
    # Convert to list if it's not already
    if not isinstance(bootstrap_effects, list):
        bootstrap_effects = [bootstrap_effects]
    
    # Convert to numeric and filter out None/NaN
    effects = []
    for e in bootstrap_effects:
        try:
            val = float(e)
            if not np.isnan(val):
                effects.append(val)
        except (ValueError, TypeError):
            continue
    
    if len(effects) == 0:
        plt.text(0.5, 0.5, "No valid bootstrap estimates", 
                ha='center', va='center', fontsize=14)
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        return plt.gcf()
    
    # Calculate statistics
    mean = np.mean(effects)
    std = np.std(effects)
    ci_lower = mean - 1.96 * std
    ci_upper = mean + 1.96 * std
    
    # Plot histogram
    plt.hist(effects, bins=20, alpha=0.5, density=True)
    
    # Add vertical lines
    plt.axvline(mean, color='blue', linestyle='--', label=f'Mean: {mean:.4f}')
    
    if true_effect is not None:
        plt.axvline(true_effect, color='red', linestyle='-', label=f'True: {true_effect:.4f}')
    
    # Add confidence interval
    plt.axvspan(ci_lower, ci_upper, alpha=0.2, color='blue', 
                label=f'95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]')
    
    plt.xlabel('Effect Size')
    plt.ylabel('Density')
    plt.title(title if title else 'Bootstrap Effect Distribution')
    plt.legend()
    plt.grid(True)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return plt.gcf()

# Select interesting parameter values for each study
interesting_cases = [
    (auto_df, 'auto', auto_df['param_value'].min()),      # Low autocorrelation
    (auto_df, 'auto', auto_df['param_value'].median()),   # Medium autocorrelation
    (auto_df, 'auto', auto_df['param_value'].max()),      # High autocorrelation
    (cross_df, 'cross', cross_df['param_value'].min()),   # Low cross-link strength
    (cross_df, 'cross', cross_df['param_value'].median()), # Medium cross-link strength
    (cross_df, 'cross', cross_df['param_value'].max()),   # High cross-link strength
    (noise_df, 'noise', noise_df['param_value'].min()),   # Low noise
    (noise_df, 'noise', noise_df['param_value'].median()), # Medium noise
    (noise_df, 'noise', noise_df['param_value'].max())    # High noise
]

for df, param_name, param_value in interesting_cases:
    # Find the closest value in the dataframe
    closest_idx = (df['param_value'] - param_value).abs().idxmin()
    closest_row = df.iloc[closest_idx]
    
    if 'bootstrap_effects' in closest_row:
        # Plot bootstrap distribution
        title = f"{param_name.capitalize()} = {closest_row['param_value']:.2f}"
        fig = plot_custom_bootstrap_histogram(
            closest_row['bootstrap_effects'],
            true_effect=closest_row['true_effect'],
            title=title,
            save_path=os.path.join(results_folder, f"bootstrap_dist_{param_name}_{closest_row['param_value']:.2f}.png")
        )
        plt.close(fig)

Analyzing bootstrap confidence interval coverage...
Generating bootstrap distribution examples...


In [6]:
for df, param_name, param_value in interesting_cases:
    # Find the closest value in the dataframe
    closest_idx = (df['param_value'] - param_value).abs().idxmin()
    closest_row = df.iloc[closest_idx]
    
    if 'bootstrap_effects' in closest_row:
        # Plot bootstrap distribution
        print("if-true")
        title = f"{param_name.capitalize()} = {closest_row['param_value']:.2f}"
        fig = plot_custom_bootstrap_histogram(
            closest_row['bootstrap_effects'],
            true_effect=closest_row['true_effect'],
            title=title,
            save_path=os.path.join(results_folder, f"bootstrap_dist_{param_name}_{closest_row['param_value']:.2f}.png")
        )
        plt.close(fig)

if-true
if-true
if-true
if-true
if-true
if-true
if-true
if-true
if-true


In [7]:
# ----------------------------------------------------------------
# Extended Analysis for Bootstrap Causal Effect Estimation
# ----------------------------------------------------------------

# Add these imports if not already present
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from mpl_toolkits.mplot3d import Axes3D
import scipy.stats as scipy_stats
import warnings

# Suppress specific deprecation warnings
warnings.filterwarnings("ignore", category=FutureWarning, message="use_inf_as_na option is deprecated")
warnings.filterwarnings("ignore", category=FutureWarning, message="When grouping with a length-1 list-like")

# Create a section for the extended analysis
print("\n" + "="*80)
print("EXTENDED ANALYSIS OF BOOTSTRAP CAUSAL EFFECT ESTIMATION")
print("="*80)

# ----------------------------------------------------------------
# 1. Enhanced Bootstrap Distribution Analysis
# ----------------------------------------------------------------
def analyze_bootstrap_distributions(df, param_name, results_folder):
    """
    Perform a detailed analysis of bootstrap effect distributions.
    
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing bootstrap effect data
    param_name : str
        Name of the parameter study being analyzed
    results_folder : str
        Folder to save results
        
    Returns
    -------
    dict
        Dictionary with bootstrap distribution metrics
    """
    # Check if we have bootstrap effects
    if 'bootstrap_effects' not in df.columns:
        print(f"No bootstrap effects found for {param_name}")
        return None
    
    # Dictionary to store results
    results = {}
    param_values = sorted(df['param_value'].unique())
    
    # Select a few representative parameter values
    if len(param_values) > 3:
        selected_values = [param_values[0], param_values[len(param_values)//2], param_values[-1]]
    else:
        selected_values = param_values
    
    # Create figure for multimodality analysis
    fig, axes = plt.subplots(len(selected_values), 2, figsize=(15, 5*len(selected_values)))
    
    # If only one value, convert to 2D array for consistent indexing
    if len(selected_values) == 1:
        axes = np.array([axes])
    
    # For each parameter value
    for i, param_val in enumerate(selected_values):
        row = df[df['param_value'] == param_val].iloc[0]
        effects = row['bootstrap_effects']
        
        # Filter out None/NaN values
        if effects is None:
            continue
            
        effects = [e for e in effects if e is not None and not np.isnan(e)]
        
        if len(effects) < 10:
            continue
        
        effects = np.array(effects)
        
        # Distribution statistics
        mean = np.mean(effects)
        median = np.median(effects)
        std = np.std(effects)
        skew = scipy_stats.skew(effects)
        kurtosis = scipy_stats.kurtosis(effects)
        
        # Check for multimodality
        kde = gaussian_kde(effects)
        x = np.linspace(mean - 3*std, mean + 3*std, 1000)
        y = kde(x)
        
        # Find peaks (local maxima)
        peaks = []
        for j in range(1, len(x) - 1):
            if y[j] > y[j-1] and y[j] > y[j+1]:
                # Only count as a peak if it's at least 20% of max height
                if y[j] > 0.2 * max(y):
                    peaks.append((x[j], y[j]))
        
        is_multimodal = len(peaks) > 1
        
        # Store results
        results[param_val] = {
            'mean': mean,
            'median': median,
            'std': std,
            'skew': skew,
            'kurtosis': kurtosis,
            'multimodal': is_multimodal,
            'num_peaks': len(peaks),
            'peaks': peaks
        }
        
        # Plot histogram and KDE
        ax1 = axes[i, 0]
        ax1.hist(effects, bins=20, density=True, alpha=0.5)
        ax1.plot(x, y, 'r-', label='KDE')
        
        # Add vertical lines for key points
        ax1.axvline(mean, color='blue', linestyle='--', label=f'Mean: {mean:.4f}')
        ax1.axvline(median, color='green', linestyle=':', label=f'Median: {median:.4f}')
        
        # Plot true effect if available
        if 'true_effect' in row and not np.isnan(row['true_effect']):
            ax1.axvline(row['true_effect'], color='red', linestyle='-', 
                      label=f'True Effect: {row["true_effect"]:.4f}')
        
        # Add 95% confidence interval
        ci_lower = mean - 1.96 * std
        ci_upper = mean + 1.96 * std
        ax1.axvspan(ci_lower, ci_upper, alpha=0.2, color='blue', 
                  label=f'95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]')
        
        # Mark peaks if multimodal
        for px, py in peaks:
            ax1.plot(px, py, 'ro')
            
        multimodal_text = "Multimodal" if is_multimodal else "Unimodal"
        ax1.set_title(f"{param_name.capitalize()} = {param_val:.2f} - {multimodal_text}")
        ax1.set_xlabel('Effect Size')
        ax1.set_ylabel('Density')
        ax1.legend()
        
        # QQ plot to check normality
        ax2 = axes[i, 1]
        scipy_stats.probplot(effects, plot=ax2)
        ax2.set_title(f"Q-Q Plot - Skew: {skew:.2f}, Kurtosis: {kurtosis:.2f}")
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_folder, f"{param_name}_bootstrap_analysis.png"), 
               dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    # Create summary dataframe
    summary_df = pd.DataFrame([
        {
            'param_value': pv,
            'distribution_mean': r['mean'],
            'distribution_median': r['median'],
            'distribution_std': r['std'],
            'distribution_skew': r['skew'],
            'distribution_kurtosis': r['kurtosis'],
            'multimodal': r['multimodal'],
            'num_peaks': r['num_peaks']
        }
        for pv, r in results.items()
    ])
    
    # Save summary dataframe
    summary_df.to_csv(os.path.join(results_folder, f"{param_name}_bootstrap_summary.csv"), index=False)
    
    return results

# ----------------------------------------------------------------
# 2. Adjustment Set Analysis
# ----------------------------------------------------------------
def analyze_adjustment_sets(results_df):
    """Analyze adjustment sets from results dataframe."""
    # Check if adjustment_set_size column exists
    if 'adjustment_set_size' not in results_df.columns and not any('adjustment_set_size' in col for col in results_df.columns):
        print("No adjustment set information found in results")
        return None
    
    # Extract adjustment set columns
    adj_cols = [col for col in results_df.columns if 'adjustment_set_size' in col]
    
    if not adj_cols:
        return None
    
    # Group by parameter value and calculate statistics
    adj_stats = results_df.groupby('param_value').agg({
        col: ['mean', 'std', 'min', 'max'] for col in adj_cols
    }).reset_index()
    
    return adj_stats

# ----------------------------------------------------------------
# 3. Graph Diversity Analysis
# ----------------------------------------------------------------
def analyze_graph_diversity(results_df):
    """Analyze graph diversity metrics from results dataframe."""
    # Check if we have graph diversity metrics
    diversity_cols = [col for col in results_df.columns if 'graph_diversity' in col or 'unique_graphs' in col]
    
    if not diversity_cols:
        print("No graph diversity metrics found in results")
        return None
    
    # Group by parameter value and calculate statistics
    diversity_stats = results_df.groupby('param_value').agg({
        col: ['mean', 'std', 'min', 'max'] for col in diversity_cols
    }).reset_index()
    
    return diversity_stats

# ----------------------------------------------------------------
# 4. Relationships Between Metrics
# ----------------------------------------------------------------
def analyze_relationships(all_results, results_folder):
    """
    Analyze relationships between graph diversity, confidence intervals, and errors.
    
    Parameters
    ----------
    all_results : pandas.DataFrame
        Combined DataFrame with all results
    results_folder : str
        Folder to save results
    """
    # Check if we have the necessary columns
    diversity_cols = [col for col in all_results.columns if 'graph_diversity' in col or 'unique_graphs' in col]
    ci_cols = ['bootstrap_ci_lower', 'bootstrap_ci_upper', 'ci_covers_true']
    error_cols = ['pcmci_abs_error', 'bagged_abs_error', 'bootstrap_abs_error']
    
    if not diversity_cols or not all(col in all_results.columns for col in ci_cols) or not all(col in all_results.columns for col in error_cols):
        print("Missing columns required for relationship analysis")
        return
    
    # Add CI width
    all_results['ci_width'] = all_results['bootstrap_ci_upper'] - all_results['bootstrap_ci_lower']
    
    # Create scatter plot matrix
    relationship_vars = ['unique_graphs_ratio', 'ci_width', 'bootstrap_abs_error', 
                        'adjustment_set_size_bootstrap'] if 'adjustment_set_size_bootstrap' in all_results.columns else ['unique_graphs_ratio', 'ci_width', 'bootstrap_abs_error']
    
    # Drop rows with NaN in any of these columns
    plot_data = all_results[['param_name'] + relationship_vars].dropna()
    
    if len(plot_data) < 3:
        print("Not enough data points for relationship analysis")
        return
    
    # Create pair plot
    g = sns.pairplot(plot_data, hue='param_name', diag_kind='kde', 
                    plot_kws={'alpha': 0.6})
    g.fig.suptitle('Relationships Between Graph Diversity, CI Width, and Error', y=1.05)
    plt.savefig(os.path.join(results_folder, "relationship_pairplot.png"), 
               dpi=300, bbox_inches='tight')
    plt.close()
    
    # Calculate correlation matrix
    corr = plot_data[relationship_vars].corr()
    
    # Plot correlation matrix
    plt.figure(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, 
               mask=mask, fmt='.2f', linewidths=0.5)
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.savefig(os.path.join(results_folder, "correlation_matrix.png"), 
               dpi=300, bbox_inches='tight')
    plt.close()
    
    # Create 3D scatter plot for main relationships
    if len(relationship_vars) >= 3:
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection='3d')
        
        # Get unique parameter names and a color for each
        param_names = plot_data['param_name'].unique()
        colors = plt.cm.tab10(np.linspace(0, 1, len(param_names)))
        
        for i, param in enumerate(param_names):
            param_data = plot_data[plot_data['param_name'] == param]
            ax.scatter(param_data['unique_graphs_ratio'], 
                     param_data['ci_width'], 
                     param_data['bootstrap_abs_error'],
                     color=colors[i], label=param, s=50, alpha=0.7)
        
        ax.set_xlabel('Graph Diversity (Unique Ratio)')
        ax.set_ylabel('CI Width')
        ax.set_zlabel('Absolute Error')
        ax.legend()
        ax.set_title('3D Relationship: Diversity, CI Width and Error')
        
        plt.savefig(os.path.join(results_folder, "3d_relationship.png"), 
                   dpi=300, bbox_inches='tight')
        plt.close()

# ----------------------------------------------------------------
# 5. LaTeX Table Generation
# ----------------------------------------------------------------
def create_bootstrap_summary_table(auto_file, cross_file, noise_file, results_folder):
    """
    Create a summary LaTeX table for bootstrap distribution properties.
    
    Parameters
    ----------
    auto_file, cross_file, noise_file : str
        Paths to bootstrap summary CSV files
    results_folder : str
        Folder to save LaTeX table
    """
    # Load summary data
    try:
        auto_summary = pd.read_csv(auto_file)
        cross_summary = pd.read_csv(cross_file)
        noise_summary = pd.read_csv(noise_file)
    except FileNotFoundError:
        print("One or more summary files not found")
        return
    
    # Create LaTeX table
    latex_table = r"\begin{table}[htbp]" + "\n"
    latex_table += r"\centering" + "\n"
    latex_table += r"\caption{Bootstrap Distribution Properties}" + "\n"
    latex_table += r"\label{tab:bootstrap_distribution}" + "\n"
    latex_table += r"\begin{tabular}{llccccc}" + "\n"
    latex_table += r"\hline" + "\n"
    latex_table += r"Parameter & Value & Mean & Median & Std & Skewness & Kurtosis \\" + "\n"
    latex_table += r"\hline" + "\n"
    
    # Add data for autocorrelation
    for _, row in auto_summary.iterrows():
        latex_table += f"Autocorrelation & {row['param_value']:.2f} & "
        latex_table += f"{row['distribution_mean']:.4f} & {row['distribution_median']:.4f} & "
        latex_table += f"{row['distribution_std']:.4f} & {row['distribution_skew']:.4f} & "
        latex_table += f"{row['distribution_kurtosis']:.4f}"
        latex_table += r" \\" + "\n"
    
    # Add data for cross-link strength
    for _, row in cross_summary.iterrows():
        latex_table += f"Cross-link & {row['param_value']:.2f} & "
        latex_table += f"{row['distribution_mean']:.4f} & {row['distribution_median']:.4f} & "
        latex_table += f"{row['distribution_std']:.4f} & {row['distribution_skew']:.4f} & "
        latex_table += f"{row['distribution_kurtosis']:.4f}"
        latex_table += r" \\" + "\n"
    
    # Add data for noise level
    for _, row in noise_summary.iterrows():
        latex_table += f"Noise & {row['param_value']:.2f} & "
        latex_table += f"{row['distribution_mean']:.4f} & {row['distribution_median']:.4f} & "
        latex_table += f"{row['distribution_std']:.4f} & {row['distribution_skew']:.4f} & "
        latex_table += f"{row['distribution_kurtosis']:.4f}"
        latex_table += r" \\" + "\n"
    
    latex_table += r"\hline" + "\n"
    latex_table += r"\end{tabular}" + "\n"
    latex_table += r"\end{table}" + "\n"
    
    # Save LaTeX table
    with open(os.path.join(results_folder, "bootstrap_distribution_table.tex"), 'w') as f:
        f.write(latex_table)
    
    print(f"Bootstrap distribution table saved to {results_folder}")

def create_graph_diversity_table(all_results, results_folder):
    """
    Create a LaTeX table for graph diversity metrics.
    
    Parameters
    ----------
    all_results : pandas.DataFrame
        Combined DataFrame with all results
    results_folder : str
        Folder to save LaTeX table
    """
    # Check if we have diversity metrics
    diversity_cols = [col for col in all_results.columns if 'graph_diversity' in col or 'unique_graphs' in col]
    if not diversity_cols:
        print("No graph diversity metrics found in results")
        return
    
    # Group by parameter name and value, calculate mean diversity metrics
    if 'unique_graphs_ratio' in all_results.columns:
        diversity_stats = all_results.groupby(['param_name', 'param_value']).agg({
            'unique_graphs_ratio': ['mean', 'std'],
            'num_unique_graphs': ['mean', 'std'] if 'num_unique_graphs' in all_results.columns else ['mean'],
            'bootstrap_abs_error': ['mean', 'std']
        }).reset_index()
        
        # Create LaTeX table
        latex_table = r"\begin{table}[htbp]" + "\n"
        latex_table += r"\centering" + "\n"
        latex_table += r"\caption{Graph Diversity and Effect Estimation Accuracy}" + "\n"
        latex_table += r"\label{tab:graph_diversity}" + "\n"
        latex_table += r"\begin{tabular}{llcccc}" + "\n"
        latex_table += r"\hline" + "\n"
        latex_table += r"Parameter & Value & \multicolumn{2}{c}{Unique Graph Ratio} & \multicolumn{2}{c}{Abs. Error} \\" + "\n"
        latex_table += r" & & Mean & Std & Mean & Std \\" + "\n"
        latex_table += r"\hline" + "\n"
        
        # Add data for each parameter study
        for param_name in ['auto', 'cross', 'noise']:
            param_data = diversity_stats[diversity_stats['param_name'] == param_name]
            
            # Sort by parameter value
            param_data = param_data.sort_values('param_value')
            
            for _, row in param_data.iterrows():
                param_label = {'auto': 'Autocorrelation', 
                         'cross': 'Cross-link', 
                         'noise': 'Noise'}.get(param_name, param_name)
            
            latex_table += f"{param_label} & {row['param_value']:.2f} & "
            
            # Add PCMCI stats
            if ('adjustment_set_size_pcmci', 'mean') in row:
                latex_table += f"{row[('adjustment_set_size_pcmci', 'mean')]:.2f} & {row[('adjustment_set_size_pcmci', 'std')]:.2f} & "
            else:
                latex_table += r"-- & -- & "
            
            # Add Bagged stats
            if ('adjustment_set_size_bagged', 'mean') in row:
                latex_table += f"{row[('adjustment_set_size_bagged', 'mean')]:.2f} & {row[('adjustment_set_size_bagged', 'std')]:.2f} & "
            else:
                latex_table += r"-- & -- & "
            
            # Add Bootstrap stats
            if ('adjustment_set_size_bootstrap', 'mean') in row:
                latex_table += f"{row[('adjustment_set_size_bootstrap', 'mean')]:.2f} & {row[('adjustment_set_size_bootstrap', 'std')]:.2f}"
            else:
                latex_table += r"-- & --"
            
            latex_table += r" \\" + "\n"
    
    latex_table += r"\hline" + "\n"
    latex_table += r"\end{tabular}" + "\n"
    latex_table += r"\end{table}" + "\n"
    
    # Save LaTeX table
    with open(os.path.join(results_folder, "adjustment_set_table.tex"), 'w') as f:
        f.write(latex_table)
    
    print(f"Adjustment set table saved to {results_folder}")



EXTENDED ANALYSIS OF BOOTSTRAP CAUSAL EFFECT ESTIMATION


In [10]:

# ----------------------------------------------------------------
# RUN THE EXTENDED ANALYSIS
# ----------------------------------------------------------------
print("Running extended analysis...")

# 1. Perform detailed bootstrap distribution analysis
print("\n1. Analyzing bootstrap distributions...")
auto_bootstrap_results = analyze_bootstrap_distributions(auto_df, 'auto', results_folder)
cross_bootstrap_results = analyze_bootstrap_distributions(cross_df, 'cross', results_folder)
noise_bootstrap_results = analyze_bootstrap_distributions(noise_df, 'noise', results_folder)

# 2. Analyze adjustment sets if available
print("\n2. Analyzing adjustment sets...")
if any('adjustment_set_size' in col for col in all_results.columns):
    # Plot adjustment set sizes for each parameter
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    
    # Autocorrelation
    auto_adj_stats = analyze_adjustment_sets(auto_df)
    if auto_adj_stats is not None:
        x = auto_adj_stats['param_value']
        for i, col in enumerate([c for c in auto_adj_stats.columns if 'adjustment_set_size' in str(c)]):
            if ('mean') in col:
                method = col[0].split('_')[-1] if isinstance(col, tuple) else col.split('_')[-1]
                axs[0].errorbar(x, auto_adj_stats[col], 
                             yerr=auto_adj_stats[col[0], 'std'] if isinstance(col, tuple) else 0, 
                             label=method.capitalize(), marker=['o', 's', '^'][i % 3])
        
        axs[0].set_xlabel('Autocorrelation')
        axs[0].set_ylabel('Adjustment Set Size')
        axs[0].set_title('Adjustment Sets - Autocorrelation')
        axs[0].legend()
        axs[0].grid(True)
    
    # Cross-link
    cross_adj_stats = analyze_adjustment_sets(cross_df)
    if cross_adj_stats is not None:
        x = cross_adj_stats['param_value']
        for i, col in enumerate([c for c in cross_adj_stats.columns if 'adjustment_set_size' in str(c)]):
            if ('mean') in col:
                method = col[0].split('_')[-1] if isinstance(col, tuple) else col.split('_')[-1]
                axs[1].errorbar(x, cross_adj_stats[col], 
                              yerr=cross_adj_stats[col[0], 'std'] if isinstance(col, tuple) else 0, 
                              label=method.capitalize(), marker=['o', 's', '^'][i % 3])
        
        axs[1].set_xlabel('Cross-link Strength')
        axs[1].set_ylabel('Adjustment Set Size')
        axs[1].set_title('Adjustment Sets - Cross-link')
        axs[1].legend()
        axs[1].grid(True)
    
    # Noise
    noise_adj_stats = analyze_adjustment_sets(noise_df)
    if noise_adj_stats is not None:
        x = noise_adj_stats['param_value']
        for i, col in enumerate([c for c in noise_adj_stats.columns if 'adjustment_set_size' in str(c)]):
            if ('mean') in col:
                method = col[0].split('_')[-1] if isinstance(col, tuple) else col.split('_')[-1]
                axs[2].errorbar(x, noise_adj_stats[col], 
                              yerr=noise_adj_stats[col[0], 'std'] if isinstance(col, tuple) else 0, 
                              label=method.capitalize(), marker=['o', 's', '^'][i % 3])
        
        axs[2].set_xlabel('Noise Level')
        axs[2].set_ylabel('Adjustment Set Size')
        axs[2].set_title('Adjustment Sets - Noise')
        axs[2].legend()
        axs[2].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_folder, "all_adjustment_sets.png"), dpi=300, bbox_inches='tight')
    plt.close()

# 3. Analyze graph diversity if available
print("\n3. Analyzing graph diversity...")
if any('unique_graphs' in col for col in all_results.columns):
    # Plot graph diversity metrics
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    
    # Autocorrelation
    auto_div_stats = analyze_graph_diversity(auto_df)
    if auto_div_stats is not None:
        x = auto_div_stats['param_value']
        for i, col in enumerate([c for c in auto_div_stats.columns if 'unique_graphs' in str(c)]):
            if ('mean') in col:
                metric = col[0].split('_')[-1] if isinstance(col, tuple) else col.split('_')[-1]
                axs[0].errorbar(x, auto_div_stats[col], 
                              yerr=auto_div_stats[col[0], 'std'] if isinstance(col, tuple) else 0, 
                              label=metric.capitalize(), marker=['o', 's'][i % 2])
        
        axs[0].set_xlabel('Autocorrelation')
        axs[0].set_ylabel('Graph Diversity')
        axs[0].set_title('Graph Diversity - Autocorrelation')
        axs[0].legend()
        axs[0].grid(True)
    
    # Cross-link
    cross_div_stats = analyze_graph_diversity(cross_df)
    if cross_div_stats is not None:
        x = cross_div_stats['param_value']
        for i, col in enumerate([c for c in cross_div_stats.columns if 'unique_graphs' in str(c)]):
            if ('mean') in col:
                metric = col[0].split('_')[-1] if isinstance(col, tuple) else col.split('_')[-1]
                axs[1].errorbar(x, cross_div_stats[col], 
                              yerr=cross_div_stats[col[0], 'std'] if isinstance(col, tuple) else 0, 
                              label=metric.capitalize(), marker=['o', 's'][i % 2])
        
        axs[1].set_xlabel('Cross-link Strength')
        axs[1].set_ylabel('Graph Diversity')
        axs[1].set_title('Graph Diversity - Cross-link')
        axs[1].legend()
        axs[1].grid(True)
    
    # Noise
    noise_div_stats = analyze_graph_diversity(noise_df)
    if noise_div_stats is not None:
        x = noise_div_stats['param_value']
        for i, col in enumerate([c for c in noise_div_stats.columns if 'unique_graphs' in str(c)]):
            if ('mean') in col:
                metric = col[0].split('_')[-1] if isinstance(col, tuple) else col.split('_')[-1]
                axs[2].errorbar(x, noise_div_stats[col], 
                              yerr=noise_div_stats[col[0], 'std'] if isinstance(col, tuple) else 0, 
                              label=metric.capitalize(), marker=['o', 's'][i % 2])
        
        axs[2].set_xlabel('Noise Level')
        axs[2].set_ylabel('Graph Diversity')
        axs[2].set_title('Graph Diversity - Noise')
        axs[2].legend()
        axs[2].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_folder, "all_graph_diversity.png"), dpi=300, bbox_inches='tight')
    plt.close()

# 4. Analyze relationships between metrics
print("\n4. Analyzing relationships between metrics...")
if 'unique_graphs_ratio' in all_results.columns and 'bootstrap_ci_lower' in all_results.columns:
    analyze_relationships(all_results, results_folder)
"""
# 5. Generate LaTeX tables for thesis
print("\n5. Generating LaTeX tables...")

# Bootstrap distribution table
auto_summary_file = os.path.join(results_folder, "auto_bootstrap_summary.csv")
cross_summary_file = os.path.join(results_folder, "cross_bootstrap_summary.csv")
noise_summary_file = os.path.join(results_folder, "noise_bootstrap_summary.csv")

if os.path.exists(auto_summary_file) and os.path.exists(cross_summary_file) and os.path.exists(noise_summary_file):
    create_bootstrap_summary_table(auto_summary_file, cross_summary_file, noise_summary_file, results_folder)
else:
    print("Bootstrap summary files not found. Run bootstrap analysis first.")

# Graph diversity table
if any('unique_graphs' in col for col in all_results.columns):
    create_graph_diversity_table(all_results, results_folder)

# Adjustment set table
if any('adjustment_set_size' in col for col in all_results.columns):
    create_adjustment_set_table(all_results, results_folder)

print("\nExtended analysis complete! All results saved to", results_folder)
"""


Running extended analysis...

1. Analyzing bootstrap distributions...

2. Analyzing adjustment sets...

3. Analyzing graph diversity...

4. Analyzing relationships between metrics...


'\n# 5. Generate LaTeX tables for thesis\nprint("\n5. Generating LaTeX tables...")\n\n# Bootstrap distribution table\nauto_summary_file = os.path.join(results_folder, "auto_bootstrap_summary.csv")\ncross_summary_file = os.path.join(results_folder, "cross_bootstrap_summary.csv")\nnoise_summary_file = os.path.join(results_folder, "noise_bootstrap_summary.csv")\n\nif os.path.exists(auto_summary_file) and os.path.exists(cross_summary_file) and os.path.exists(noise_summary_file):\n    create_bootstrap_summary_table(auto_summary_file, cross_summary_file, noise_summary_file, results_folder)\nelse:\n    print("Bootstrap summary files not found. Run bootstrap analysis first.")\n\n# Graph diversity table\nif any(\'unique_graphs\' in col for col in all_results.columns):\n    create_graph_diversity_table(all_results, results_folder)\n\n# Adjustment set table\nif any(\'adjustment_set_size\' in col for col in all_results.columns):\n    create_adjustment_set_table(all_results, results_folder)\n\nprin

In [ ]:

def create_adjustment_set_table(all_results, results_folder):
    """
    Create a LaTeX table for adjustment set statistics.
    
    Parameters
    ----------
    all_results : pandas.DataFrame
        Combined DataFrame with all results
    results_folder : str
        Folder to save LaTeX table
    """
    # Check if we have adjustment set data
    adjustment_cols = [col for col in all_results.columns if 'adjustment_set_size' in col]
    if not adjustment_cols:
        print("No adjustment set data found in results")
        return
    
    # Group by parameter name and value, calculate stats
    adj_stats = all_results.groupby(['param_name', 'param_value']).agg({
        'adjustment_set_size_pcmci': ['mean', 'std', 'min', 'max'] if 'adjustment_set_size_pcmci' in all_results.columns else ['mean'],
        'adjustment_set_size_bagged': ['mean', 'std', 'min', 'max'] if 'adjustment_set_size_bagged' in all_results.columns else ['mean'],
        'adjustment_set_size_bootstrap': ['mean', 'std', 'min', 'max'] if 'adjustment_set_size_bootstrap' in all_results.columns else ['mean']
    }).reset_index()
    
    # Create LaTeX table
    latex_table = r"\begin{table}[htbp]" + "\n"
    latex_table += r"\centering" + "\n"
    latex_table += r"\caption{Adjustment Set Sizes by Method}" + "\n"
    latex_table += r"\label{tab:adjustment_sets}" + "\n"
    latex_table += r"\begin{tabular}{llcccccc}" + "\n"
    latex_table += r"\hline" + "\n"
    latex_table += r"Parameter & Value & \multicolumn{2}{c}{PCMCI} & \multicolumn{2}{c}{Bagged} & \multicolumn{2}{c}{Bootstrap} \\" + "\n"
    latex_table += r" & & Mean & Std & Mean & Std & Mean & Std \\" + "\n"
    latex_table += r"\hline" + "\n"
    
    # Add data for each parameter study
    for param_name in ['auto', 'cross', 'noise']:
        param_data = adj_stats[adj_stats['param_name'] == param_name]
        
        # Sort by parameter value
        param_data = param_data.sort_values('param_value')
        
        for _, row in param_data.iterrows():
            param_label = {'auto': 'Autocorrelation', 
                         'cross': 'Cross-link', 
                         'noise': 'Noise'}.get(param_name, param_name)
            
            latex_table += f"{param_label} & {row['param_value']:.2f} & "
            
            # Add PCMCI stats
            if ('adjustment_set_size_pcmci', 'mean') in row:
                latex_table += f"{row[('adjustment_set_size_pcmci', 'mean')]:.2f} & {row[('adjustment_set_size_pcmci', 'std')]:.2f} & "
            else:
                latex_table += r"-- & -- & "
            
            # Add Bagged stats
            if ('adjustment_set_size_bagged', 'mean') in row:
                latex_table += f"{row[('adjustment_set_size_bagged', 'mean')]:.2f} & {row[('adjustment_set_size_bagged', 'std')]:.2f} & "
            else:
                latex_table += r"-- & -- & "
            
            # Add Bootstrap stats
            if ('adjustment_set_size_bootstrap', 'mean') in row:
                latex_table += f"{row[('adjustment_set_size_bootstrap', 'mean')]:.2f} & {row[('adjustment_set_size_bootstrap', 'std')]:.2f}"
            else:
                latex_table += r"-- & --"
            
            latex_table += r" \\" + "\n"
    
    latex_table += r"\hline" + "\n"
    latex_table += r"\end{tabular}" + "\n"
    latex_table += r"\end{table}" + "\n"
    
    # Save LaTeX table
    with open(os.path.join(results_folder, "adjustment_set_table.tex"), 'w') as f:
        f.write(latex_table)
    
    print(f"Adjustment set table saved to {results_folder}")

# Generate the tables if bootstrap analysis files exist
print("Generating LaTeX tables for thesis...")

# Bootstrap distribution table
auto_summary_file = os.path.join(results_folder, "auto_bootstrap_summary.csv")
cross_summary_file = os.path.join(results_folder, "cross_bootstrap_summary.csv")
noise_summary_file = os.path.join(results_folder, "noise_bootstrap_summary.csv")

if os.path.exists(auto_summary_file) and os.path.exists(cross_summary_file) and os.path.exists(noise_summary_file):
    create_bootstrap_summary_table(auto_summary_file, cross_summary_file, noise_summary_file, results_folder)
else:
    print("Bootstrap summary files not found. Run bootstrap analysis first.")

# Graph diversity table
create_graph_diversity_table(all_results, results_folder)

# Adjustment set table
create_adjustment_set_table(all_results, results_folder)

In [ ]:

# ------------------------------------------------------------------
# Generate LaTeX tables
# ------------------------------------------------------------------
print("Generating LaTeX tables...")

# Function to create LaTeX table from metrics DataFrame
def create_latex_table(metrics_df, param_name):
    # Parameter labels
    param_labels = {
        'auto': 'Autocorrelation',
        'cross': 'Cross-link Strength',
        'noise': 'Noise Level'
    }
    
    # Select representative rows (min, median, max)
    values = sorted(metrics_df['param_value'].unique())
    selected_values = [values[0], values[len(values)//2], values[-1]]
    selected_metrics = metrics_df[metrics_df['param_value'].isin(selected_values)]
    
    # Create LaTeX table
    latex_table = f"\\begin{{table}}[htbp]\n"
    latex_table += f"\\centering\n"
    latex_table += f"\\caption{{Effect Estimation Performance for Varying {param_labels.get(param_name, param_name)}}}\n"
    latex_table += f"\\label{{tab:{param_name}_performance}}\n"
    latex_table += f"\\begin{{tabular}}{{l|ccc|ccc|c}}\n"
    latex_table += f"\\hline\n"
    latex_table += f"{param_labels.get(param_name, param_name)} & \\multicolumn{{3}}{{c|}}{{MAE}} & \\multicolumn{{3}}{{c|}}{{RMSE}} & CI Coverage \\\\\n"
    latex_table += f" Value & PCMCI & Bagged & Bootstrap & PCMCI & Bagged & Bootstrap & Bootstrap \\\\\n"
    latex_table += f"\\hline\n"
    
    for _, row in selected_metrics.iterrows():
        latex_table += f"{row['param_value']:.2f} & "
        latex_table += f"{row.get('pcmci_mae', float('nan')):.4f} & {row.get('bagged_mae', float('nan')):.4f} & {row.get('bootstrap_mae', float('nan')):.4f} & "
        latex_table += f"{row.get('pcmci_rmse', float('nan')):.4f} & {row.get('bagged_rmse', float('nan')):.4f} & {row.get('bootstrap_rmse', float('nan')):.4f} & "
        
        if 'bootstrap_ci_coverage' in row:
            latex_table += f"{row['bootstrap_ci_coverage']*100:.1f}\\% \\\\\n"
        else:
            latex_table += f"N/A \\\\\n"
    
    latex_table += f"\\hline\n"
    latex_table += f"\\end{{tabular}}\n"
    latex_table += f"\\end{{table}}\n"
    
    return latex_table

# Generate and save tables
auto_table = create_latex_table(auto_metrics, 'auto')
cross_table = create_latex_table(cross_metrics, 'cross')
noise_table = create_latex_table(noise_metrics, 'noise')

with open(os.path.join(results_folder, "auto_table.tex"), 'w') as f:
    f.write(auto_table)
    
with open(os.path.join(results_folder, "cross_table.tex"), 'w') as f:
    f.write(cross_table)
    
with open(os.path.join(results_folder, "noise_table.tex"), 'w') as f:
    f.write(noise_table)

# Create consolidated table with overall results
def create_consolidated_table():
    # Calculate overall metrics for each parameter study
    overall_metrics = pd.DataFrame([
        {'Parameter': 'Autocorrelation', **calculate_metrics(auto_df).iloc[0]},
        {'Parameter': 'Cross-link Strength', **calculate_metrics(cross_df).iloc[0]},
        {'Parameter': 'Noise Level', **calculate_metrics(noise_df).iloc[0]}
    ])
    
    # Calculate improvement percentages
    for i, row in overall_metrics.iterrows():
        pcmci_mae = row.get('pcmci_mae', 0)
        bagged_mae = row.get('bagged_mae', 0)
        bootstrap_mae = row.get('bootstrap_mae', 0)
        
        if pcmci_mae > 0:
            overall_metrics.loc[i, 'bagged_mae_imp'] = 100 * (pcmci_mae - bagged_mae) / pcmci_mae
            overall_metrics.loc[i, 'bootstrap_mae_imp'] = 100 * (pcmci_mae - bootstrap_mae) / pcmci_mae
    
    # Create LaTeX table
    latex_table = f"\\begin{{table}}[htbp]\n"
    latex_table += f"\\centering\n"
    latex_table += f"\\caption{{Summary of Effect Estimation Performance Across Parameter Studies}}\n"
    latex_table += f"\\label{{tab:overall_performance}}\n"
    latex_table += f"\\begin{{tabular}}{{l|ccc|cc|c}}\n"
    latex_table += f"\\hline\n"
    latex_table += f"Parameter & \\multicolumn{{3}}{{c|}}{{MAE}} & \\multicolumn{{2}}{{c|}}{{Improvement (\\%)}} & CI Coverage \\\\\n"
    latex_table += f"Study & PCMCI & Bagged & Bootstrap & Bagged & Bootstrap & Bootstrap \\\\\n"
    latex_table += f"\\hline\n"
    
    for _, row in overall_metrics.iterrows():
        latex_table += f"{row['Parameter']} & "
        latex_table += f"{row.get('pcmci_mae', float('nan')):.4f} & {row.get('bagged_mae', float('nan')):.4f} & {row.get('bootstrap_mae', float('nan')):.4f} & "
        latex_table += f"{row.get('bagged_mae_imp', float('nan')):.1f}\\% & {row.get('bootstrap_mae_imp', float('nan')):.1f}\\% & "
        
        if 'bootstrap_ci_coverage' in row:
            latex_table += f"{row['bootstrap_ci_coverage']*100:.1f}\\% \\\\\n"
        else:
            latex_table += f"N/A \\\\\n"
    
    latex_table += f"\\hline\n"
    latex_table += f"\\end{{tabular}}\n"
    latex_table += f"\\end{{table}}\n"
    
    return latex_table, overall_metrics

# Generate and save consolidated table
consolidated_table, overall_metrics = create_consolidated_table()
with open(os.path.join(results_folder, "consolidated_table.tex"), 'w') as f:
    f.write(consolidated_table)

# Save overall metrics
overall_metrics.to_csv(os.path.join(results_folder, "overall_metrics.csv"), index=False)

print(f"\nAnalysis complete! Results saved to {results_folder}/")